# xAI: dynamic credentials from Vault

`./demo.sh` includes these stages. The plugin is bundled in the `josemerchan/vault-xai:2.2.0-ent` image published on Docker Hub; it requires neither a build nor a local copy of its repository. The management key comes in through `XAI_MANAGEMENT_API_KEY`, is kept in `verify/secrets/xai_mgmt` and configures `verify/xai`.

The application uses the **Client Credentials** of `chatbot-xai` to read `xai/creds/lumen?ttl=300`. It does not read `secrets/xai`. On an empty installation the role has a 5 min TTL, a 10 min maximum and native expiry; on an existing one the user-configured role is preserved and checked to allow the requested TTL.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/xai.sh


## Configuration without displaying keys


In [ ]:
%%bash
source verify/scripts/common.sh
vault read -format=json xai/config | jq '.data|{api_endpoint,team_id}'
vault read -format=json xai/roles/lumen | jq '.data|{acls,ttl,max_ttl,set_native_expiry}'


## Identity of the `chatbot-xai` agent

Requires the Verify applications and users from [notebook 6](6_Lumen_Chatbot_Vault_IAM.ipynb) (`02-verify-apps.sh`): the `vault-chatbot-xai` application is created from the `chatbot/research` template. It registers the agent with mandatory RAR and selects xAI as the model provider.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/configure-xai.sh


## Test inference and real expiry

The test issues a disposable 60 s key, calls Grok, checks denials and waits for it to expire. It revokes only its own lease. It does not print the key, does not revoke by prefix and does not affect the keys Lumen is using.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/verify-xai.sh


The current plugin does not implement renewal. Lumen reuses a key while it covers the execution budget and requests another when 210 s or less remain. Vault revokes the previous one when it expires and xAI applies its native expiry. See [lifecycle and traces](chatbot/README.md#xai-con-claves-dinámicas).
